# Evaluation with GroUSE-style Judges

Grade grounded answers on completeness, usefulness and faithfulness, and check that the *judge* behaves.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

GroUSE (Grounded QA Evaluation) scores answers on several criteria and, importantly, **unit-tests the judge** with deliberately flawed answers. Here: answer relevancy, completeness, usefulness and faithfulness on a 1-5 scale.

In [ ]:
CRITERIA = {
    "answer_relevancy": "Does the answer address the question without off-topic content? (5 = fully on topic)",
    "completeness": "Does the answer include all the relevant information from the context? (5 = nothing missing)",
    "usefulness": "Is the answer helpful and clear for a user? (5 = very helpful)",
    "faithfulness": "Is every claim supported by the context, with nothing invented? (5 = fully supported)",
}

def judge(question, contexts, answer_text):
    scores = {}
    for name, rubric in CRITERIA.items():
        out = llm(
            f"You are a strict grader.\nCriterion: {rubric}\n\nContext:\n" + "\n".join(contexts) +
            f"\n\nQuestion: {question}\nAnswer: {answer_text}\n\nReply with a single integer from 1 to 5."
        )
        m = re.search(r"[1-5]", out)
        scores[name] = int(m.group()) if m else None
    return scores

## Judge unit tests
A trustworthy judge must punish known-bad answers. We build a good answer and three degraded versions.

In [ ]:
q = "How many dimensions does all-MiniLM-L6-v2 produce?"
ctx = [h["text"] for h in store.search(q, 2)]
variants = {
    "good":        "The all-MiniLM-L6-v2 model produces 384-dimensional vectors.",
    "incomplete":  "It produces vectors.",
    "hallucinated": "It produces 1536-dimensional vectors and was trained by OpenAI on code.",
    "off_topic":   "Paris is the capital of France.",
}
import pandas as pd
pd.DataFrame({name: judge(q, ctx, a) for name, a in variants.items()}).T

Expect `good` to score highest everywhere, `hallucinated` to fail **faithfulness**, `incomplete` to fail **completeness** and `off_topic` to fail **relevancy**. If your judge cannot tell these apart, switch to a stronger judge model.